# Kural — English SER: frozen Whisper encoder + attention pooling + classifier

Same design as the Tamil notebook, but on **English** (CREMA-D, 7,442 clips, 91 actors) —
Whisper's strongest language, so the frozen features should carry more emotion. Tests
whether the earlier weak Tamil scores were the *language* or the *frozen-ASR-features*.

Frozen encoder + learnable attention pooling (masks 30s padding) + linear classifier;
only attention + classifier train. **Speaker-independent** split (held-out actors).
GPU runtime + Internet; token via Kaggle/Colab Secret `HF_TOKEN`. All datasets are ungated.

In [ ]:
!pip -q install --upgrade "transformers>=4.44" "datasets[audio]>=2.20" "accelerate>=0.33" scikit-learn librosa soundfile huggingface_hub

In [ ]:
# ===== CONFIG =====
BACKBONE     = "openai/whisper-small"
HF_USERNAME  = "Venky0411"
HUB_REPO     = f"{HF_USERNAME}/whisper-small-en-emotion"

# CREMA-D has 6 (angry/disgust/fear/happy/neutral/sad). Default to 4 for a clean comparison.
LABELS       = ["angry", "happy", "neutral", "sad"]
label2id     = {l: i for i, l in enumerate(LABELS)}
id2label     = {i: l for l, i in label2id.items()}

TEST_FRAC        = 0.2    # fraction of ACTORS held out for the speaker-independent test
USE_AUG          = False  # add RAVDESS + TESS to TRAIN (-> ~12k clips)
N_HEADS          = 8
EPOCHS           = 20
BATCH_SIZE       = 16
LR               = 1e-3

In [ ]:
from huggingface_hub import login
import os
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    pass
if not HF_TOKEN:
    try:  # Kaggle: Add-ons -> Secrets -> HF_TOKEN (toggle ON for this notebook)
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or os.environ.get('HF_TOKEN')
if HF_TOKEN:
    login(token=HF_TOKEN); print('HF login OK — will push to the Hub')
else:
    print('No HF_TOKEN found -> training runs but will NOT push. '
          'Add a Kaggle Secret named HF_TOKEN (Add-ons -> Secrets) and enable it to push.')


In [ ]:
# ===== loaders =====
import os
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
from datasets import load_dataset, Audio, concatenate_datasets

EMO_MAP = {"anger": "angry", "angry": "angry", "mad": "angry",
           "happy": "happy", "happiness": "happy", "joy": "happy",
           "sad": "sad", "sadness": "sad",
           "neutral": "neutral", "calm": "neutral", "normal": "neutral",
           "fear": "fear", "fearful": "fear",
           "disgust": "disgust", "disgusted": "disgust",
           "surprise": None, "surprised": None, "ps": None, "pleasant surprise": None}
def canon(v):
    return None if v is None else EMO_MAP.get(str(v).strip().lower().replace("_", " "))

def load_cremad():
    parts = []
    for sp in ["train", "validation", "test"]:
        try:
            parts.append(load_dataset("confit/cremad-parquet", split=sp, verification_mode="no_checks"))
        except Exception:
            pass
    ds = concatenate_datasets(parts)
    def _m(b):
        b["emo"] = canon(b.get("emotion"))
        b["speaker"] = os.path.basename(str(b.get("file", ""))).split("_")[0] or "unk"
        b["source"] = "cremad"
        return b
    ds = ds.map(_m).filter(lambda b: b["emo"] is not None)
    ds = ds.cast_column("audio", Audio(sampling_rate=16000)).select_columns(["audio", "emo", "speaker", "source"])
    print(f"CREMA-D: {len(ds)} clips, {len(set(ds['speaker']))} actors")
    return ds

def load_norm(name, source, config=None):
    ds = load_dataset(name, config, split="train", verification_mode="no_checks")
    cands = [c for c in ["label", "labels", "emotion", "emotions", "class"] if c in ds.column_names]
    lc = next((c for c in cands if ds.features[c].__class__.__name__ == "ClassLabel"), cands[0] if cands else None)
    feat = ds.features[lc] if lc else None
    is_cl = feat.__class__.__name__ == "ClassLabel" if feat is not None else False
    def _m(b):
        v = b[lc] if lc else None
        if is_cl and isinstance(v, int):
            v = feat.int2str(v)
        b["emo"] = canon(v); b["source"] = source
        return b
    ds = ds.map(_m).filter(lambda b: b["emo"] is not None)
    return ds.cast_column("audio", Audio(sampling_rate=16000)).select_columns(["audio", "emo", "source"])

In [ ]:
# ===== load CREMA-D + select classes + speaker-independent split (by actor) =====
from collections import Counter
keep = set(LABELS)
cremad = load_cremad()
cremad = cremad.filter(lambda b: b["emo"] in keep)

actors = sorted(set(cremad["speaker"]))
n_test = max(1, int(len(actors) * TEST_FRAC))
test_actors = set(actors[-n_test:])
print(f"actors: {len(actors)} | held-out test actors: {len(test_actors)}")

test_ds  = cremad.filter(lambda b: b["speaker"] in test_actors).remove_columns("speaker")
train_c  = cremad.filter(lambda b: b["speaker"] not in test_actors).remove_columns("speaker")

aug = []
if USE_AUG:
    for name, src, cfg in [("confit/ravdess-parquet", "ravdess", "fold1"), ("AbstractTTS/TESS", "tess", None)]:
        try:
            d = load_norm(name, src, cfg); aug.append(d); print(f"  aug OK {src}: {len(d)}")
        except Exception as e:
            print(f"  aug SKIP {src}: {type(e).__name__}")

train_ds = concatenate_datasets([train_c] + aug).filter(lambda b: b["emo"] in keep).shuffle(seed=42)
print("train:", len(train_ds), "| test:", len(test_ds))
print("train labels:", dict(Counter(train_ds["emo"])))
print("test  labels:", dict(Counter(test_ds["emo"])))

In [ ]:
# ===== feature extraction: log-mel + attention mask =====
from transformers import WhisperFeatureExtractor
fe = WhisperFeatureExtractor.from_pretrained(BACKBONE)

def prep(b):
    out = fe(b["audio"]["array"], sampling_rate=16000, return_attention_mask=True)
    b["input_features"] = out.input_features[0]
    b["attention_mask"] = out.attention_mask[0]
    b["labels"] = label2id[b["emo"]]
    return b

train_ds = train_ds.map(prep, remove_columns=train_ds.column_names, num_proc=2)
test_ds = test_ds.map(prep, remove_columns=test_ds.column_names, num_proc=2)

In [ ]:
# ===== model: frozen Whisper encoder -> attention pooling -> classifier =====
import torch, torch.nn as nn
from transformers import WhisperModel

class WhisperAttnSER(nn.Module):
    def __init__(self, backbone, num_labels, n_heads=8, dropout=0.1):
        super().__init__()
        self.encoder = WhisperModel.from_pretrained(backbone).encoder
        for p in self.encoder.parameters():
            p.requires_grad = False                       # FROZEN encoder
        d = self.encoder.config.d_model
        self.query = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        self.attn = nn.MultiheadAttention(d, n_heads, batch_first=True)
        self.norm = nn.LayerNorm(d)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d, num_labels)

    def forward(self, input_features, attention_mask=None, labels=None):
        self.encoder.eval()
        with torch.no_grad():
            enc = self.encoder(input_features).last_hidden_state
        kpm = None
        if attention_mask is not None:
            m = attention_mask[:, ::2][:, :enc.size(1)]
            kpm = (m == 0)
            kpm[kpm.all(dim=1)] = False
        q = self.query.expand(enc.size(0), -1, -1)
        pooled, _ = self.attn(q, enc, enc, key_padding_mask=kpm)
        pooled = self.norm(pooled.squeeze(1))
        logits = self.classifier(self.dropout(pooled))
        out = {"logits": logits}
        if labels is not None:
            out["loss"] = nn.functional.cross_entropy(logits, labels)
        return out

model = WhisperAttnSER(BACKBONE, num_labels=len(LABELS), n_heads=N_HEADS)
print("trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"accuracy": accuracy_score(p.label_ids, preds),
            "f1_macro": f1_score(p.label_ids, preds, average="macro")}

In [ ]:
from transformers import TrainingArguments, Trainer, default_data_collator

args = TrainingArguments(
    output_dir="./whisper-en-emotion",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    warmup_ratio=0.1,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    remove_unused_columns=False,
    report_to=["none"],
    push_to_hub=bool(HF_TOKEN),
    hub_model_id=HUB_REPO,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=test_ds,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
# ===== speaker-independent test report =====
from sklearn.metrics import classification_report, confusion_matrix
pred = trainer.predict(test_ds)
y, yhat = pred.label_ids, pred.predictions.argmax(1)
print(classification_report(y, yhat, target_names=LABELS, digits=3))
print("confusion matrix (rows=true):\n", confusion_matrix(y, yhat))

In [ ]:
# ===== save + push =====
import json
os.makedirs("./whisper-en-emotion", exist_ok=True)
torch.save(model.state_dict(), "./whisper-en-emotion/pytorch_model.bin")
json.dump({"backbone": BACKBONE, "labels": LABELS, "n_heads": N_HEADS, "arch": "WhisperAttnSER"},
          open("./whisper-en-emotion/ser_config.json", "w"))
fe.save_pretrained("./whisper-en-emotion")
try:
    trainer.push_to_hub()
    print("Pushed:", HUB_REPO)
except Exception as e:
    print("push skipped:", e)